Notebook này  dùng để mở lại DB đã build, embedding câu hỏi bằng OpenAI, kiểm tra top_k chunk.

# **0. Cài đặt thư viện**

In [ ]:
# Cài các thư viện cần thiết nếu máy chưa có.
!pip install -q openai chromadb pandas

# **1. Import thư viện và khai báo cấu hình**

In [ ]:
import os
from typing import Any, Dict, List, Optional, Tuple
from dotenv import load_dotenv

import chromadb
from openai import OpenAI

# ============================================================
# CẤU HÌNH CHUNG
# ============================================================

# Thư mục Chroma DB đã được tạo ở Notebook 01.
# Nếu bạn chạy file 01 và file 02 cùng thư mục thì giữ nguyên đường dẫn này.
CHROMA_DIR = "./chroma_ou_rag_db_openai"

# Tên collection phải giống với Notebook 01.
COLLECTION_NAME = "ou_academic_rag_openai"

# Model embedding phải giống model đã dùng khi build DB ở Notebook 01.
# Nếu Notebook 01 dùng text-embedding-3-small thì Notebook 02 cũng phải dùng đúng model này.
EMBEDDING_MODEL = "text-embedding-3-small"

# Đọc biến môi trường từ file .env trong cùng thư mục project
load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY đã sẵn sàng.")
else:
    print("Bạn chưa set OPENAI_API_KEY.")

# Khởi tạo OpenAI client để gọi embedding API.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Đã khởi tạo OpenAI client.")

# **2. Mở Chroma collection**

In [ ]:
# Đọc DB từ thư mục đã tạo ở Notebook 01.
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Nếu lỗi ở đây: thường là chưa chạy Notebook 01 hoặc sai CHROMA_DIR.
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Số item trong collection:", collection.count())

# **3. Hàm tạo embedding cho câu hỏi**

In [ ]:
def embed_query(query: str) -> List[float]:
    """
    Tạo embedding cho câu hỏi người dùng.

    Yêu cầu quan trọng:
    - Query phải dùng cùng EMBEDDING_MODEL với document chunks.
    - Nếu document dùng text-embedding-3-small thì query cũng phải dùng text-embedding-3-small.
    """
    # Câu hỏi phải embed bằng cùng model với documents.
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query
    )
    return response.data[0].embedding

# **4. Hàm retrieve cơ bản**

In [ ]:
def retrieve(
    query: str,
    top_k: int = 5,
    where: Optional[Dict[str, Any]] = None,
    include_distance: bool = True
) -> List[Dict[str, Any]]:
    """
    Retrieval cơ bản: tìm top_k chunk gần nhất với câu hỏi.

    Tham số:
    - query: câu hỏi người dùng.
    - top_k: số chunk muốn lấy.
    - where: filter metadata của Chroma, ví dụ {"document_type": "tuition"}.

    Khi nào dùng where?
    - Câu hỏi học phí       -> where={"document_type": "tuition"}
    - Câu hỏi CTĐT          -> where={"document_type": "curriculum"}
    - Câu hỏi kế hoạch năm  -> where={"document_type": "academic_plan"}
    - Câu hỏi sổ tay        -> where={"document_type": "student_handbook"}
    """
    # 1. Embed câu hỏi thành vector.
    query_embedding = embed_query(query)

    # 2. Query Chroma bằng vector câu hỏi.
    # Chroma sẽ so sánh query_embedding với embedding của các chunks đã lưu.
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"]
    )

    # 3. Chuẩn hóa kết quả Chroma thành list dict dễ đọc/dễ xử lý.
    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "rank": i + 1,
            "id": results["ids"][0][i],
            "distance": results["distances"][0][i],
            "document": results["documents"][0][i],
            "metadata": results["metadatas"][0][i]
        })

    return hits


def infer_where_filter(query: str) -> Optional[Dict[str, Any]]:
    """
    Suy luận filter metadata dựa trên câu hỏi.

    Mục tiêu:
    - Giảm nhiễu retrieval.
    - Ví dụ câu hỏi học phí không nên tìm lẫn trong CTĐT hoặc sổ tay nếu đã có document_type=tuition.

    Lưu ý:
    - Đây là rule đơn giản, phục vụ đồ án và demo.
    - Nếu rule sai, có thể chỉnh từ khóa bên dưới.
    """
    q = query.lower()

    # Nhóm học phí: ưu tiên tài liệu học phí.
    if any(k in q for k in ["học phí", "mức phí", "tín chỉ bao nhiêu", "đóng phí"]):
        return {"document_type": "tuition"}

    # Nhóm kế hoạch đào tạo năm học / mốc thời gian / xét tốt nghiệp trong kế hoạch.
    if any(k in q for k in ["kế hoạch xét tốt nghiệp", "mốc thời gian", "năm học 2025", "kế hoạch đào tạo năm học"]):
        return {"document_type": "academic_plan"}

    # Nhóm CTĐT: học phần, tín chỉ toàn khóa, PLO, chuẩn đầu ra chương trình.
    if any(k in q for k in ["chương trình đào tạo", "ctđt", "học phần", "tín chỉ", "plo", "chuẩn đầu ra"]):
        # Ngoại lệ: các câu quy chế có thể chứa chữ tín chỉ/chuẩn đầu ra nhưng không hỏi CTĐT.
        if not any(k in q for k in ["cảnh báo", "buộc thôi học", "bảo lưu", "nghỉ học", "xét tốt nghiệp"]):
            return {"document_type": "curriculum"}

    # Nhóm sổ tay: phòng ban, liên hệ, thủ tục, xác nhận, kênh hỗ trợ.
    if any(k in q for k in ["liên hệ", "phòng ban", "thủ tục", "xác nhận", "kênh hỗ trợ", "hỗ trợ sinh viên", "xem thông tin học tập"]):
        return {"document_type": "student_handbook"}

    # Nhóm quy chế/sổ tay đào tạo: để None để retrieval tự tìm trong cả regulation và handbook.
    return None


def keyword_boost_score(query: str, hit: Dict[str, Any]) -> float:
    """
    Tính điểm boost thủ công để sắp xếp lại kết quả sau khi Chroma retrieve.

    Vì semantic search đôi khi lấy đúng document_type nhưng sai chunk_type,
    ta thêm một số rule nhẹ:
    - Hỏi PLO -> ưu tiên chunk_type='plo'.
    - Hỏi học phần -> ưu tiên course_table.
    - Hỏi tổng tín chỉ -> ưu tiên program_info.
    - Hỏi CNTT -> ưu tiên document_name chứa CongNgheThongTin.
    - Hỏi học phí -> ưu tiên tuition_table.
    - Hỏi buộc thôi học -> ưu tiên đúng mục "2. Buộc thôi học".
    """
    q = query.lower()
    meta = hit.get("metadata", {}) or {}
    doc_name = str(meta.get("document_name", ""))
    doc_type = str(meta.get("document_type", ""))
    chunk_type = str(meta.get("chunk_type", ""))
    section = str(meta.get("section", ""))
    text = str(hit.get("document", ""))[:3000].lower()
    section_l = section.lower()

    score = 0.0

    # Ưu tiên đúng ngành Công nghệ thông tin khi câu hỏi có nhắc ngành này.
    if "công nghệ thông tin" in q:
        if "CongNgheThongTin" in doc_name or "công nghệ thông tin" in text:
            score += 3.0
        # Giảm điểm nếu là CTĐT ngành khác.
        if doc_type == "curriculum" and "CongNgheThongTin" not in doc_name:
            score -= 2.0

    # Hỏi PLO/chuẩn đầu ra chương trình thì ưu tiên chunk PLO trong CTĐT.
    if "plo" in q or "chuẩn đầu ra" in q:
        if chunk_type == "plo":
            score += 5.0
        if doc_type == "student_handbook" and "tin học" in text:
            score -= 3.0

    # Hỏi danh sách học phần/học phần cơ sở ngành thì ưu tiên bảng học phần.
    if "học phần" in q or "môn" in q or "cơ sở ngành" in q:
        if chunk_type == "course_table":
            score += 4.0
        if chunk_type == "course_description":
            score += 1.0

    # Hỏi tổng số tín chỉ toàn khóa thì thường nằm trong program_info.
    if "tổng số tín chỉ" in q or "bao nhiêu tín chỉ" in q or "tích lũy bao nhiêu tín chỉ" in q:
        if chunk_type == "program_info":
            score += 5.0
        if "total credits" in text or "tổng số tín chỉ" in text:
            score += 3.0

    # Hỏi kế hoạch học kỳ thì ưu tiên section có kế hoạch đào tạo/học kỳ.
    if "học kỳ" in q or "học kì" in q:
        if "kế hoạch đào tạo" in text or "học kỳ" in text or "họckỳ" in text:
            score += 3.0

    # Học phí thì ưu tiên tài liệu tuition_table.
    if "học phí" in q:
        if doc_type == "tuition":
            score += 5.0
        if chunk_type == "tuition_table":
            score += 5.0
        if "chương trình chuẩn" in q and "chương trình chuẩn" in text:
            score += 3.0
        if "công nghệ thông tin" in q and "công nghệ thông tin" in text:
            score += 2.0

    # Kế hoạch xét tốt nghiệp trong năm học thì ưu tiên academic_plan.
    if "kế hoạch xét tốt nghiệp" in q or "nộp đơn đề nghị xét tốt nghiệp" in q:
        if doc_type == "academic_plan":
            score += 4.0
        if "kế hoạch xét tốt nghiệp" in text:
            score += 4.0

    # Tinh chỉnh 1: câu hỏi "buộc thôi học".
    # Mục đúng nhất là section "2. Buộc thôi học" hoặc đoạn có heading này.
    # Giảm nhẹ phần "Buộc nghỉ học tạm thời" vì nó cùng Điều 28 nhưng không trực tiếp trả lời.
    if "buộc thôi học" in q:
        if "2. buộc thôi học" in section_l or "## 2. buộc thôi học" in text:
            score += 7.0
        elif "buộc thôi học" in section_l or "buộc thôi học" in text:
            score += 3.0
        if "buộc nghỉ học tạm thời" in section_l or "## 1. buộc nghỉ học tạm thời" in text:
            score -= 2.0

    return score


def get_adaptive_retrieval_params(query: str) -> Dict[str, int]:
    """
    Tinh chỉnh 2: tự chọn top_k, raw_top_k và số ký tự preview theo loại câu hỏi.

    Vì sao cần?
    - PLO: mỗi PLO là một chunk riêng, top_k=5 có thể chưa đủ để xem hết.
    - Học phí: bảng học phí dài, preview 700 ký tự có thể cắt mất số tiền.
    - Các câu còn lại: top_k=5 và preview 700 là đủ dễ đọc.
    """
    q = query.lower()

    if "plo" in q or "chuẩn đầu ra" in q:
        return {"top_k": 10, "raw_top_k": 30, "preview_chars": 900}

    if "học phí" in q:
        return {"top_k": 5, "raw_top_k": 20, "preview_chars": 1500}

    return {"top_k": 5, "raw_top_k": 20, "preview_chars": 700}


def retrieve_routed(query: str, top_k: int = 5, raw_top_k: int = 20) -> List[Dict[str, Any]]:
    """
    Retrieval cải tiến có routing/filter + rerank nhẹ.

    Bước xử lý:
    1. Tự suy luận document_type phù hợp bằng infer_where_filter().
    2. Retrieve nhiều hơn top_k một chút, ví dụ raw_top_k=20.
    3. Rerank lại bằng distance + keyword_boost_score().
    4. Trả về top_k kết quả tốt nhất.

    Vì sao cần raw_top_k?
    - Nếu lấy đúng 5 kết quả ban đầu, chunk đúng có thể nằm rank 6-10.
    - Lấy 20 rồi rerank giúp kéo chunk đúng lên trên mà không cần rebuild DB.
    """
    where = infer_where_filter(query)

    # Retrieve lần 1 với filter tự động.
    try:
        candidates = retrieve(query=query, top_k=raw_top_k, where=where)
    except Exception as e:
        print("Retrieval có filter bị lỗi, fallback sang retrieval không filter.")
        print("where =", where)
        print("error =", e)
        candidates = retrieve(query=query, top_k=raw_top_k, where=None)

    # Nếu filter quá hẹp và không có kết quả, fallback sang toàn bộ DB.
    if not candidates:
        candidates = retrieve(query=query, top_k=raw_top_k, where=None)

    # Rerank: Chroma distance càng nhỏ càng tốt, boost càng lớn càng tốt.
    # final_score càng nhỏ càng tốt.
    reranked = []
    for hit in candidates:
        boost = keyword_boost_score(query, hit)
        final_score = float(hit["distance"]) - 0.05 * boost
        new_hit = dict(hit)
        new_hit["auto_where"] = where
        new_hit["keyword_boost"] = boost
        new_hit["final_score"] = final_score
        reranked.append(new_hit)

    reranked.sort(key=lambda x: x["final_score"])

    # Cập nhật lại rank sau rerank.
    top_hits = reranked[:top_k]
    for i, hit in enumerate(top_hits):
        hit["rank"] = i + 1

    return top_hits


# **5. Hàm in kết quả retrieval**

In [ ]:
def print_hits(hits: List[Dict[str, Any]], preview_chars: int = 900) -> None:
    """
    In kết quả retrieval theo format dễ đọc.

    Nên xem các field sau:
    - document_type: có đúng nhóm tài liệu không?
    - chunk_type: có đúng loại chunk không?
    - document_name: có đúng file/ngành không?
    - Preview: đoạn text có đủ thông tin để trả lời không?
    """
    for hit in hits:
        meta = hit["metadata"]
        print("=" * 100)
        print(f"Rank {hit['rank']} | distance = {hit['distance']:.4f}")

        # Nếu dùng retrieve_routed(), các thông tin này sẽ xuất hiện.
        if "auto_where" in hit:
            print("auto_where:", hit.get("auto_where"))
            print("keyword_boost:", hit.get("keyword_boost"), "| final_score:", round(hit.get("final_score", 0), 4))

        print("ID:", hit["id"])
        print("document_type:", meta.get("document_type"))
        print("chunk_type:", meta.get("chunk_type"))
        print("document_name:", meta.get("document_name"))
        print("page:", meta.get("page_start"), "-", meta.get("page_end"))
        print("section:", meta.get("section"))
        print("article:", meta.get("article"))
        print("Preview:")
        print(hit["document"][:preview_chars])


# **6. Test với nhiều câu hỏi khác nhau (mẫu)**

## **6.1. Test với bộ câu hỏi ngắn**

In [ ]:
test_questions_short = [
    # Regulation / quy chế
    "Sinh viên cần điều kiện gì để được xét tốt nghiệp?",
    "Khi nào sinh viên bị cảnh báo học tập?",
    "Sinh viên được xin nghỉ học tạm thời và bảo lưu kết quả học tập trong những trường hợp nào?",
    "Sinh viên bị buộc thôi học trong những trường hợp nào?",

    # Student handbook / sổ tay
    "Sinh viên liên hệ đơn vị nào để được hỗ trợ đăng ký môn học, thời khóa biểu, lịch thi và kế hoạch đào tạo?",
    "Sinh viên làm thủ tục xác nhận vay vốn hoặc xác nhận sinh viên như thế nào?",

    # Curriculum / chương trình đào tạo
    "Tổng số tín chỉ toàn khóa của chương trình đào tạo ngành Công nghệ thông tin là bao nhiêu?",
    "Chuẩn đầu ra PLO của chương trình đào tạo ngành Công nghệ thông tin gồm những nội dung nào?",

    # Academic plan / kế hoạch năm học
    "Kế hoạch xét tốt nghiệp trong năm học gồm những mốc nào?",

    # Tuition / học phí
    "Mức học phí bình quân của nhóm ngành Công nghệ thông tin chương trình chuẩn là bao nhiêu?"
]

for question in test_questions_short:
    print("" + "#" * 120)
    print("QUESTION:", question)

    params = get_adaptive_retrieval_params(question)

    hits = retrieve_routed(
        question,
        top_k=params["top_k"],
        raw_top_k=params["raw_top_k"]
    )

    print_hits(hits, preview_chars=params["preview_chars"])


## **6.2. So sánh retrieval cơ bản và retrieval cải tiến trên một câu khó**

Các câu hỏi như học phí, PLO, học phần ngành CNTT thường dễ bị retrieval lạc sang tài liệu khác nếu không filter.

In [ ]:
hard_question = "Chuẩn đầu ra PLO của chương trình đào tạo ngành Công nghệ thông tin gồm những nội dung nào?"

# Lấy tham số adaptive riêng cho câu hỏi khó này.
params = get_adaptive_retrieval_params(hard_question)

print("#" * 120)
print("RETRIEVAL CƠ BẢN")
print("QUESTION:", hard_question)

basic_hits = retrieve(
    hard_question,
    top_k=params["top_k"]
)

print_hits(
    basic_hits,
    preview_chars=params["preview_chars"]
)

print("#" * 120)
print("RETRIEVAL CẢI TIẾN: ROUTING + RERANK")
print("QUESTION:", hard_question)

routed_hits = retrieve_routed(
    hard_question,
    top_k=params["top_k"],
    raw_top_k=params["raw_top_k"]
)

print_hits(
    routed_hits,
    preview_chars=params["preview_chars"]
)


## **6.3. Test riêng từng nhóm tài liệu bằng filter thủ công**

In [ ]:
manual_tests = [
    {
        "question": "Học phí một tín chỉ là bao nhiêu?",
        "where": {"document_type": "tuition"}
    },
    {
        "question": "Mức học phí của ngành Công nghệ thông tin là bao nhiêu?",
        "where": {"document_type": "tuition"}
    },
    {
        "question": "Tổng số tín chỉ toàn khóa của chương trình đào tạo ngành Công nghệ thông tin là bao nhiêu?",
        "where": {"document_type": "curriculum"}
    },
    {
        "question": "Kế hoạch xét tốt nghiệp trong năm học gồm những mốc nào?",
        "where": {"document_type": "academic_plan"}
    },
    {
        "question": "Sinh viên liên hệ đơn vị nào để được hỗ trợ đăng ký môn học, thời khóa biểu, lịch thi và kế hoạch đào tạo?",
        "where": {"document_type": "student_handbook"}
    }
]

for item in manual_tests:
    question = item["question"]
    where = item["where"]

    print("\n" + "#" * 120)
    print("QUESTION:", question)
    print("WHERE:", where)

    params = get_adaptive_retrieval_params(question)

    hits = retrieve(
        question,
        top_k=params["top_k"],
        where=where
    )

    print_hits(
        hits,
        preview_chars=params["preview_chars"]
    )


# **7. Bộ câu hỏi dài để đánh giá kỹ hơn**

Chạy cell này sau khi bộ 10 câu đã ổn. Output sẽ dài hơn, phù hợp để bạn ghi nhận kết quả trong báo cáo đồ án.

In [ ]:
test_questions_long = [
    # ============================================================
    # 1. Regulation / Quy chế đào tạo
    # Mục tiêu: kiểm tra các quy định học vụ cốt lõi
    # ============================================================
    "Sinh viên cần đáp ứng những điều kiện nào để được xét và công nhận tốt nghiệp?",
    "Sinh viên bị cảnh báo học tập khi điểm trung bình học kỳ ở mức nào?",
    "Sinh viên được xin nghỉ học tạm thời và bảo lưu kết quả học tập trong những trường hợp nào?",
    "Sinh viên bị buộc thôi học khi thuộc những trường hợp nào?",
    "Sinh viên thi hộ hoặc nhờ người thi hộ sẽ bị xử lý như thế nào?",

    # ============================================================
    # 2. Student handbook / Sổ tay sinh viên
    # Mục tiêu: kiểm tra thông tin phòng ban, thủ tục, hệ thống hỗ trợ
    # ============================================================
    "Sinh viên liên hệ Phòng Quản lý đào tạo để được hỗ trợ những vấn đề nào?",
    "Sinh viên liên hệ đơn vị nào để được hỗ trợ đăng ký môn học, thời khóa biểu, lịch thi và kế hoạch đào tạo?",
    "Sinh viên làm thủ tục xác nhận vay vốn ngân hàng chính sách xã hội như thế nào?",
    "Sinh viên có thể xem kết quả học tập cá nhân trên hệ thống nào?",
    "Sinh viên kiểm tra thời khóa biểu cá nhân và học phí ở đâu?",

    # ============================================================
    # 3. Curriculum / Chương trình đào tạo ngành Công nghệ thông tin
    # Mục tiêu: kiểm tra đúng CTĐT ngành CNTT, không lẫn ngành khác
    # ============================================================
    "Tổng số tín chỉ toàn khóa của chương trình đào tạo ngành Công nghệ thông tin là bao nhiêu?",
    "Thời gian đào tạo chuẩn của chương trình đào tạo ngành Công nghệ thông tin là bao nhiêu học kỳ?",
    "Chuẩn đầu ra PLO của chương trình đào tạo ngành Công nghệ thông tin gồm những nội dung nào?",
    "Trong chương trình đào tạo ngành Công nghệ thông tin, học phần Cơ sở lập trình có mã môn học và số tín chỉ là gì?",
    "Trong chương trình đào tạo ngành Công nghệ thông tin, học phần Kỹ thuật lập trình có điều kiện tiên quyết là gì?",

    # ============================================================
    # 4. Academic plan / Kế hoạch đào tạo năm học
    # Mục tiêu: kiểm tra đúng file kế hoạch đào tạo năm học 2025-2026
    # ============================================================
    "Trong kế hoạch đào tạo năm học 2025-2026, thời gian học tập học kỳ 1 là khi nào?",
    "Trong kế hoạch đào tạo năm học 2025-2026, kế hoạch xét tốt nghiệp gồm những đợt nào?",
    "Trong kế hoạch đào tạo năm học 2025-2026, sinh viên nộp đơn đề nghị xét KLTN hoặc ĐATN bổ sung vào thời gian nào?",
    "Trong kế hoạch đào tạo năm học 2025-2026, thời gian công bố danh sách đủ điều kiện tốt nghiệp là khi nào?",
    "Trong kế hoạch đào tạo năm học 2025-2026, thời gian nhận đơn xét miễn giảm môn học là khi nào?",

    # ============================================================
    # 5. Tuition / Học phí
    # Mục tiêu: hỏi đúng kiểu dữ liệu học phí hiện có: mức học phí bình quân
    # ============================================================
    "Mức học phí bình quân của nhóm ngành Công nghệ thông tin chương trình chuẩn là bao nhiêu?",
    "Mức học phí bình quân của nhóm ngành Công nghệ thông tin chương trình tiên tiến là bao nhiêu?",
    "Trong chương trình chuẩn, nhóm ngành Công nghệ sinh học và Công nghệ thực phẩm có mức học phí bình quân là bao nhiêu?",
    "Trong chương trình chuẩn, nhóm ngành Công nghệ thông tin gồm những ngành nào?",
    "Tài liệu học phí dự kiến khóa 2026 có phân biệt chương trình chuẩn và chương trình tiên tiến không?"
]

for question in test_questions_long:
    print("\n" + "#" * 120)
    print("QUESTION:", question)

    # Tự điều chỉnh tham số theo từng loại câu hỏi.
    # Ví dụ:
    # - PLO cần top_k lớn hơn vì mỗi PLO có thể là một chunk riêng.
    # - Học phí cần preview dài hơn để thấy đủ bảng và con số.
    params = get_adaptive_retrieval_params(question)

    hits = retrieve_routed(
        question,
        top_k=params["top_k"],
        raw_top_k=params["raw_top_k"]
    )

    print_hits(
        hits,
        preview_chars=params["preview_chars"]
    )